# 🐟 Pipeline de nettoyage — FAO Global Production Fish 2024.1.0

Notebook qui décompose le script `clean_fish.py` en étapes documentées.

## Dataset source
**FAO Global Production Quantity 2024.1.0** — `data/raw/GlobalProductionFish_2024.1.0/`

Contenu :
- `Global_production_quantity.csv` : 1.15M lignes (1950-2022)
- `CL_FI_COUNTRY_GROUPS.csv` : 247 pays avec UN/ISO2/ISO3 codes
- `CL_FI_SPECIES_GROUPS.csv` : 3591 espèces avec scientific name + groupes ISSCAAP
- `CL_FI_PRODUCTION_SOURCE_DET.csv` : 4 sources (Capture + 3 aquacultures)

## Pipeline en 9 étapes
1. Chargement (1.15M lignes brutes)
2. Filtrage MEASURE=Q_tlw (tonnes poids vivant)
3. Jointure pays UN → ISO2
4. Jointure espèces 3A → ISSCAAP + Scientific Name
5. Jointure source production
6. Détection outliers (winsorize optionnel)
7. Renommage final + sauvegarde
8. Agrégations multiples (par source, par ISSCAAP)
9. Stats finales

## Outputs
- `fish_production_raw_clean.csv` (190 MB, 1.1M lignes propres)
- `fish_production_by_source.csv` (capture/aquaculture par pays/année)
- `fish_production_total.csv` (totaux + diversité espèces)
- `fish_production_by_isscaap.csv` (par groupe biologique)

## 1. Imports & chemins

In [ ]:
import os, sys, io
import numpy as np
import pandas as pd

RAW_DIR = '../data/raw/GlobalProductionFish_2024.1.0'  # depuis sous-dossier couche1_planete
OUT_DIR = '../data/cleaned'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Files raw : {os.listdir(RAW_DIR)[:10]}')

## 2. Chargement données brutes

In [ ]:
df = pd.read_csv(f'{RAW_DIR}/Global_production_quantity.csv', low_memory=False)
print(f'{len(df):,} lignes brutes, années {df["PERIOD"].min()}–{df["PERIOD"].max()}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 3. Filtrage MEASURE=Q_tlw

Le dataset contient deux mesures :
- `Q_tlw` : Quantity in tonnes live weight (production en tonnes)
- `Q_no_1` : Number (individus pour certaines espèces)

On garde uniquement Q_tlw pour avoir une mesure homogène en tonnes.

In [ ]:
before = len(df)
df = df[df['MEASURE'] == 'Q_tlw'].copy()
df = df.drop(columns=['MEASURE'])
print(f'{len(df):,} lignes après filtre (était {before:,})')
print(f'STATUS counts: {df["STATUS"].value_counts().to_dict()}')
# A = official, I = imputed, N = national estimate, E = estimate

## 4. Jointure pays UN_Code → ISO2 standardisé

In [ ]:
cn = pd.read_csv(f'{RAW_DIR}/CL_FI_COUNTRY_GROUPS.csv', low_memory=False)
cn = cn[['UN_Code', 'ISO2_Code', 'ISO3_Code', 'Name_En',
         'Continent_Group_En', 'EcoClass_Group_En']].rename(columns={
            'UN_Code': 'COUNTRY.UN_CODE',
            'ISO2_Code': 'ISO',
            'ISO3_Code': 'ISO3',
            'Name_En': 'Pays',
            'Continent_Group_En': 'Continent',
            'EcoClass_Group_En': 'EcoClass'
         })
df = df.merge(cn, on='COUNTRY.UN_CODE', how='left')
n_no_iso = df['ISO'].isna().sum()
print(f'{n_no_iso:,} lignes sans ISO ({n_no_iso/len(df)*100:.1f}%)')
print(f'→ ce sont les agrégats régionaux FAO (ex: World, Europe...) qui n\'ont pas de code ISO')
df = df.dropna(subset=['ISO'])
print(f'\n{len(df):,} lignes avec ISO valide, {df["ISO"].nunique()} pays uniques')

## 5. Jointure espèces → ISSCAAP

ISSCAAP = International Standard Statistical Classification of Aquatic Animals and Plants.
Ex de groupes : "Marine fishes nei", "Salmons, trouts, smelts", "Cyprinids", "Shrimps, prawns"...

In [ ]:
sp = pd.read_csv(f'{RAW_DIR}/CL_FI_SPECIES_GROUPS.csv', low_memory=False)
sp = sp[['3A_Code', 'Name_En', 'Scientific_Name', 'Major_Group',
         'ISSCAAP_Group_En']].rename(columns={
            '3A_Code': 'SPECIES.ALPHA_3_CODE',
            'Name_En': 'Espece_En',
            'Major_Group': 'MajorGroup',
            'ISSCAAP_Group_En': 'ISSCAAP'
         })
df = df.merge(sp, on='SPECIES.ALPHA_3_CODE', how='left')
df['ISSCAAP'] = df['ISSCAAP'].fillna('Unknown')
df['MajorGroup'] = df['MajorGroup'].fillna('Unknown')

print(f'Top 10 groupes ISSCAAP :')
print(df['ISSCAAP'].value_counts().head(10).to_string())

## 6. Jointure source production (Capture vs Aquaculture)

In [ ]:
ps = pd.read_csv(f'{RAW_DIR}/CL_FI_PRODUCTION_SOURCE_DET.csv', low_memory=False)
ps = ps[['Code', 'Name_En']].rename(columns={'Code': 'PRODUCTION_SOURCE_DET.CODE',
                                              'Name_En': 'Source'})
df = df.merge(ps, on='PRODUCTION_SOURCE_DET.CODE', how='left')

def simplify_source(s):
    if pd.isna(s): return 'Unknown'
    s = str(s).lower()
    if 'capture' in s: return 'Capture'
    if 'freshwater' in s: return 'Aquaculture_Freshwater'
    if 'brackish' in s: return 'Aquaculture_Brackish'
    if 'marine' in s: return 'Aquaculture_Marine'
    return 'Other'
df['SourceType'] = df['Source'].apply(simplify_source)
print(df['SourceType'].value_counts().to_string())

## 7. Détection outliers et NaN

Pas de NaN dans VALUE (vérifié plus haut). Pour les outliers :
- 45% des lignes sont à 0 (espèces non pêchées certaines années) → normal
- Max = 12.3M tonnes (anchois Pérou 1971) → valide historiquement
- Pas de winsorization : les valeurs extrêmes sont réelles

In [ ]:
print(f'NaN dans VALUE : {df["VALUE"].isna().sum()}')
print(f'Zéros : {(df["VALUE"]==0).sum():,} ({(df["VALUE"]==0).mean()*100:.1f}%)')
print(f'Négatifs : {(df["VALUE"]<0).sum()}')
print(f'\nQuantiles :')
print(df['VALUE'].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_string())
print(f'\nP99.9 = {df["VALUE"].quantile(0.999):,.0f} tonnes')
print(f'Max   = {df["VALUE"].max():,.0f} tonnes (production halieutique extrême historique)')

## 8. Renommage final + sauvegarde

In [ ]:
df = df.rename(columns={
    'PERIOD': 'Annee',
    'VALUE': 'Tonnes',
    'AREA.CODE': 'Zone_FAO',
})
df = df[['ISO', 'ISO3', 'Pays', 'Continent', 'EcoClass', 'Annee',
         'SPECIES.ALPHA_3_CODE', 'Espece_En', 'Scientific_Name', 'MajorGroup', 'ISSCAAP',
         'Zone_FAO', 'SourceType', 'Source', 'Tonnes', 'STATUS']]
df.to_csv(f'{OUT_DIR}/fish_production_raw_clean.csv', index=False)
print(f'✓ {OUT_DIR}/fish_production_raw_clean.csv  ({len(df):,} lignes)')

## 9. Agrégations utiles

### A. Par source (Capture vs 3 aquacultures)

In [ ]:
df_pos = df[df['Tonnes'] > 0]
by_src = (df_pos.groupby(['ISO', 'Annee', 'SourceType'])['Tonnes']
          .sum().unstack(fill_value=0).reset_index())
by_src.columns = ['ISO', 'Annee'] + [f'fish_{c}_t' for c in by_src.columns[2:]]
print(f'Shape : {by_src.shape}')
by_src.to_csv(f'{OUT_DIR}/fish_production_by_source.csv', index=False)
by_src.head()

### B. Total + diversité d'espèces

In [ ]:
by_total = df_pos.groupby(['ISO', 'Annee']).agg(
    fish_total_t=('Tonnes', 'sum'),
    fish_species_count=('SPECIES.ALPHA_3_CODE', 'nunique'),
    fish_isscaap_count=('ISSCAAP', 'nunique'),
).reset_index()
print(f'Shape : {by_total.shape}')
by_total.to_csv(f'{OUT_DIR}/fish_production_total.csv', index=False)
by_total.head()

### C. Top 10 groupes ISSCAAP par pays/année

In [ ]:
isscaap_top = (df_pos.groupby(['ISO', 'Annee', 'ISSCAAP'])['Tonnes']
                .sum().unstack(fill_value=0).reset_index())
# Garder seulement les 10 groupes ISSCAAP les plus produits mondialement
top_groups = (df_pos.groupby('ISSCAAP')['Tonnes'].sum()
              .sort_values(ascending=False).head(10).index)
keep_cols = ['ISO', 'Annee'] + [g for g in top_groups if g in isscaap_top.columns]
isscaap_top = isscaap_top[keep_cols].copy()

def safe_col(s):
    return 'fish_isscaap_' + ''.join(c if c.isalnum() else '_' for c in s.lower())[:40]
isscaap_top.columns = ['ISO', 'Annee'] + [safe_col(c) for c in isscaap_top.columns[2:]]
print(f'Shape : {isscaap_top.shape}')
isscaap_top.to_csv(f'{OUT_DIR}/fish_production_by_isscaap.csv', index=False)
isscaap_top.head()

## 10. Stats finales

In [ ]:
print(f'Pays couverts : {by_total["ISO"].nunique()}')
print(f'Années : {by_total["Annee"].min()}–{by_total["Annee"].max()}')
print(f'Production totale 2022 : {by_total[by_total["Annee"]==2022]["fish_total_t"].sum()/1e6:.1f} M tonnes')

print(f'\nTop 10 pays 2022 :')
top2022 = by_total[by_total['Annee']==2022].sort_values('fish_total_t', ascending=False).head(10)
for _, r in top2022.iterrows():
    print(f'  {r["ISO"]}: {r["fish_total_t"]/1e6:.2f} M t  ({int(r["fish_species_count"])} espèces)')

print(f'\nOutputs créés :')
for f in ['fish_production_raw_clean.csv','fish_production_by_source.csv',
          'fish_production_total.csv','fish_production_by_isscaap.csv']:
    p = f'{OUT_DIR}/{f}'
    sz = os.path.getsize(p) / 1e6
    print(f'  {f}  ({sz:.1f} MB)')